<a href="https://colab.research.google.com/github/avnig309-dotcom/StockPricePrediction_project/blob/main/stock_prediciton_editing_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import yfinance as yf
import pandas as pd
#yfinance is a library and in this library the datasets have by default datetime as index
df = yf.download("^NSEI", period="max",auto_adjust=False)

[*********************100%***********************]  1 of 1 completed


note that date appers neeche while price adj close etc appears above this indicates that date column is used as index.

also A ticker (or ticker symbol) is a unique short code used to identify a stock, index, ETF, or other financial instrument on a stock exchange.

example:-

Microsoft->MSFT,
Tesla->Inc.	TSLA,
Reliance Industries->RELIANCE (NSE),
Tata Consultancy Services->TCS,
NIFTY 50->^NSEI (in Yahoo Finance)

RN the database is multi-index.Database i.e=>two column levels.

In single index dataframes:-print(df.columns) looks like this=>

Index(['Open', 'High', 'Low', 'Close', 'Volume'], dtype='object')

Multi-index looks like this:-

MultiIndex([('Adj Close', '^NSEI'),
            (    'Close', '^NSEI'),
            (     'High', '^NSEI'),
            (      'Low', '^NSEI'),
            (     'Open', '^NSEI'),
            (   'Volume', '^NSEI'),
            ])

now our dataframe is multi index (having two column levels) but along with that there is also a name featre added

names=['Price', 'Ticker']

isse ye pata chal rha ki jo pehla level of columns hai jisme low high etc hai uss levlel ke saare columns ka naam have been set to price

and jo dusra level hai ^NSEI wala uss level ke sare columns ka naam is ticker

note that we dont necessarily have names assigned to columns

so we are gonna remove the ticker column here coz sabka ticker is same ie ^NSEI and then we will only have one column toh column name rakhne ka bhi koi fayeda nhi hai so we will remove that as well



In [3]:
df.columns.names = [None, None]
df.columns=df.columns.droplevel(1)
print(df.iloc[50:100])

              Adj Close        Close         High          Low         Open  \
Date                                                                          
2007-11-28  5617.549805  5617.549805  5749.950195  5595.500000  5699.549805   
2007-11-29  5634.600098  5634.600098  5725.000000  5612.100098  5617.799805   
2007-11-30  5762.750000  5762.750000  5782.549805  5632.649902  5633.899902   
2007-12-03  5865.000000  5865.000000  5878.799805  5754.600098  5765.450195   
2007-12-04  5858.350098  5858.350098  5897.250000  5840.299805  5870.200195   
2007-12-05  5940.000000  5940.000000  5949.299805  5859.950195  5861.899902   
2007-12-06  5954.700195  5954.700195  6027.049805  5919.799805  5941.049805   
2007-12-07  5974.299805  5974.299805  6042.100098  5894.799805  5963.600098   
2007-12-10  5960.600098  5960.600098  6015.299805  5923.350098  5974.000000   
2007-12-11  6097.250000  6097.250000  6111.200195  5960.399902  5960.399902   
2007-12-12  6159.299805  6159.299805  6175.649902  6

Relative Strength (RS) is calculated as the average of all positive price movements (up moves) divided by the average of all negative price movements (down moves)

We'll illustrate the calculation of RSI on the example of the most common period, 14. For RSI calculation you need closing prices of the last 15 days (for RSI with a period of 10, you need the last 11 closing prices etc.).

For each bar, up move (U) equals:

Closet - Closet-1 if the price change is positive
Zero if the price change is negative or zero

Down move (D) equals:

The absolute value of Closet - Closet-1 if the price change is negative
Zero if the price change is positive or zero

Step 2: Averaging the Advances and Declines
Three different approaches are commonly used. They differ in the way how average up and down moves are calculated:

Simple Moving Average

Exponential Moving Average

Wilder's Smoothing Method

Simple Moving Average

Under this method, which is the most straightforward, AvgU and AvgD are calculated as simple moving averages:

AvgU = sum of all up moves (U) in the last N bars divided by N

AvgD = sum of all down moves (D) in the last N bars divided by N

N = RSI period

Most stock trading platforms calculate RSI using Wilder's Smoothing Method. IN this method:-

a = 1 / N

and therefore 1 - a = ( N - 1 ) / N

N = RSI period

AvgUt = a * Ut + ( 1 - a ) * AvgUt-1

AvgDt = a * Dt + ( 1 - a ) * AvgDt-1

Calculate U and D for ALL rows

Suppose your data is:

Row	Close	Diff	U	D
0	100	—	—	—
1	105	+5	5	0
2	103	-2	0	2
3	108	+5	5	0
4	106	-2	0	2
5	110	+4	4	0
...	...	...	...	...

You continue this until the last row.

So if you have 4609 rows, you calculate 4608 values of U and D.

Step 2: Initial average

Now take only the first 15 values of U.

Suppose they are

5
0
5
0
4
...

Compute

avgU = mean(first 15 U values)

Similarly,

avgD = mean(first 15 D values)

This gives the first average.

Step 3: Remaining rows

Now go to Row 15.

You do not recompute the average of another 15 values.

Instead,

avgU15 =
(1/15) × U15
+
(14/15) × avgU14

Then

Row 16

avgU16 =
(1/15) × U16
+
(14/15) × avgU15

Then

Row 17

avgU17 =
(1/15) × U17
+
(14/15) × avgU16

Continue until the end of the DataFrame.

Think of it like this
Calculate U and D
─────────────────

Row 0   ✓
Row 1   ✓
Row 2   ✓
...
Row 4608 ✓


Calculate average
─────────────────

Rows 0-14  → Simple Average

Then

Row 15 → Smoothed Average
Row 16 → Smoothed Average
Row 17 → Smoothed Average
...
Row 4608 → Smoothed Average

So:

✅ U and D are calculated for every row.
✅ Only the first average uses the first 15 values.
✅ After that, each new average is computed from the previous average and the current U or D, not by taking another 15-row average.

In [4]:
# for r in df.itertuples():#r is an entire row, not a row number.
  #.loc[row_index name, column_name] → Access by label/index name. here we would have to pass date
  #.iloc[row_position, column_position] → Access by integer position
  # diff=df.loc[r+1,"Close"]-df.loc[r,"Close"] since r is an entire row r+1 does not make any sense

df["U"] = 0.0
df["D"] = 0.0

# k=0
# while k!=len(df):
#   for i in range(k,k+15):

for i in range(len(df)):

    # diff=df.iloc[i,2]-df.iloc[i+1,2]
    diff = df["Close"].iloc[i] - df["Close"].iloc[i-1]

    # diff2=-(df.iloc[i,2]-df.iloc[i+1,2])
    diff2 = -diff

    # if df.iloc[i+1,2]<df.iloc[i,2]:
    if df["Close"].iloc[i-1] < df["Close"].iloc[i]:

        if diff > 0:

            # df.loc[df.index[i], "U"]=diff
            df.at[df.index[i], "U"] = diff

        else:

            # df.loc[df.index[i], "U"]=0
            df.at[df.index[i], "U"] = 0.0

    else:

        if diff2 > 0:

            # df.loc[df.index[i], "D"]=diff2
            df.at[df.index[i], "D"] = diff2

        else:

            # df.loc[df.index[i], "D"]=0
            df.at[df.index[i], "D"] = 0.0

#   k=k+15


# Initial averages
df["avgUt"] = df["U"].rolling(window=15).mean()
df["avgDt"] = df["D"].rolling(window=15).mean()

a = 1/15

# for i in range(15,len(df)-1):
for i in range(15, len(df)):

    # df.loc[df.index[i], "avgUt"]=a*df.iloc[i,9]+(1-a)*df.loc[df.index[i-1], "avgUt"]
    df.at[df.index[i], "avgUt"] = (
        a * df.at[df.index[i], "U"]
        + (1-a) * df.at[df.index[i-1], "avgUt"]
    )

    # df.loc[df.index[i], "avgDt"]=a*df.loc[df.index[i],"D"]+(1-a)*df.loc[df.index[i-1], "avgDt"]
    df.at[df.index[i], "avgDt"] = (
        a * df.at[df.index[i], "D"]
        + (1-a) * df.at[df.index[i-1], "avgDt"]
    )
df['RS']=df['avgUt']/df['avgDt']
df["RSI"] = 100 - (100 / (1 + df["RS"]))
df["SMA20"]=df["Close"].rolling(window=20).mean()
df["SMA50"]=df["Close"].rolling(window=50).mean()
print(df.tail())

               Adj Close         Close          High           Low  \
Date                                                                 
2026-07-03  24270.849609  24270.849609  24378.150391  24252.349609   
2026-07-06  24430.349609  24430.349609  24458.650391  24287.099609   
2026-07-07  24398.699219  24398.699219  24530.900391  24348.949219   
2026-07-08  23882.050781  23882.050781  24300.000000  23805.199219   
2026-07-09  23962.800781  23962.800781  24134.699219  23925.699219   

                    Open  Volume           U           D      avgUt  \
Date                                                                  
2026-07-03  24375.650391  326600   95.150391    0.000000  84.658223   
2026-07-06  24306.849609  329400  159.500000    0.000000  89.647675   
2026-07-07  24464.449219  412600    0.000000   31.650391  83.671163   
2026-07-08  24259.550781  420500    0.000000  516.648438  78.093086   
2026-07-09  23928.949219       0   80.750000    0.000000  78.270213   

           

In [5]:
import time

start = time.time()

s = 0
for i in range(10_000_000):
    s += i

print(time.time() - start)

1.323563575744629


In [6]:
df = df.reset_index()
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date", ascending=True)
df = df.reset_index(drop=True)

df["EMA20"] = df["Close"].ewm(span=20, adjust=False).mean()

df["EMA50"] = df["Close"].ewm(span=50, adjust=False).mean()

ema12 = df["Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Close"].ewm(span=26, adjust=False).mean()
df["MACD"] = ema12 - ema26

df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
print(df.tail())


           Date     Adj Close         Close          High           Low  \
4608 2026-07-03  24270.849609  24270.849609  24378.150391  24252.349609   
4609 2026-07-06  24430.349609  24430.349609  24458.650391  24287.099609   
4610 2026-07-07  24398.699219  24398.699219  24530.900391  24348.949219   
4611 2026-07-08  23882.050781  23882.050781  24300.000000  23805.199219   
4612 2026-07-09  23962.800781  23962.800781  24134.699219  23925.699219   

              Open  Volume           U           D      avgUt      avgDt  \
4608  24375.650391  326600   95.150391    0.000000  84.658223  55.892508   
4609  24306.849609  329400  159.500000    0.000000  89.647675  52.166341   
4610  24464.449219  412600    0.000000   31.650391  83.671163  50.798611   
4611  24259.550781  420500    0.000000  516.648438  78.093086  81.855266   
4612  23928.949219       0   80.750000    0.000000  78.270213  76.398248   

            RS        RSI         SMA20         SMA50         EMA20  \
4608  1.514661  60.23

ewm() is a pandas function used to calculate the exponential moving average (EMA). It calculates the average of past stock prices but gives more importance to recent prices and less importance to older prices

since EMA helps identify stock trends, the ewm function reacts faster to price changes compared to a normal moving average.

span=20 means we are calculating the EMA using a 20-day period.

adjust=False applies the standard recursive EMA formula where the current EMA depends on the previous EMA value.

.mean() calculates the final weighted average value.

similarly, for EMA50 the ewm() function is used in the same manner as before.

span=50 means the calculation is performed over 50 trading days.

EMA20 helps the model to identify short-term price trends, while EMA50 helps the model understand long-term trends and overall market behaviour.

For calculating the MACD

first we calculate EMA12 (over the span of 12 days) to represent short-term price movement

then we calculate the EMA26 (over a span of 26 days) which represents longer-term price movement

then we subtract the EMA26 from the EMA12 using the formula MACD = EMA12 - EMA26

MACD helps the model identify changes in momentum and trend direction by comparing short-term and long-term price movements

The MACD Signal Line helps identify possible trend changes and momentum shifts.

after calculating MACD we apply another EMA calculation to the MACD values.

macd signal line calculation: MACD Signal = EMA9(MACD)

span=9 indicates we calculate a 9-day EMA of the MACD line.
 this helps smooth out the fluctuations in MACD values and creates the signal line.

Calculation of Daily Stock Return

Daily Return=

(Close(t)-Close(t-1))/Close(t-1)*100

100 se mal for percentage answer

In [7]:
df["Daily Return(%)"]=0.0
for i in range (1,len(df)):
  # df["Daily Return(%)"].iloc[i]=((df["Close"].iloc[i]-df["Close"].iloc[i-1])/df["Close"].iloc[i-1])*100
  # do not use iloc at the lhs coz iloc is for reading only=>why?=?iloc is a slicing method and creating slice are temporary hence
  #it is okay to use iloc in rhs coz values have been already added once in the memory but not right to use it for rhs since the allocation
  #will end up being temporary
  df.at[df.index[i], "Daily Return(%)"]=((df["Close"].iloc[i]-df["Close"].iloc[i-1])/df["Close"].iloc[i-1])*100
print(df.tail())


           Date     Adj Close         Close          High           Low  \
4608 2026-07-03  24270.849609  24270.849609  24378.150391  24252.349609   
4609 2026-07-06  24430.349609  24430.349609  24458.650391  24287.099609   
4610 2026-07-07  24398.699219  24398.699219  24530.900391  24348.949219   
4611 2026-07-08  23882.050781  23882.050781  24300.000000  23805.199219   
4612 2026-07-09  23962.800781  23962.800781  24134.699219  23925.699219   

              Open  Volume           U           D      avgUt      avgDt  \
4608  24375.650391  326600   95.150391    0.000000  84.658223  55.892508   
4609  24306.849609  329400  159.500000    0.000000  89.647675  52.166341   
4610  24464.449219  412600    0.000000   31.650391  83.671163  50.798611   
4611  24259.550781  420500    0.000000  516.648438  78.093086  81.855266   
4612  23928.949219       0   80.750000    0.000000  78.270213  76.398248   

            RS        RSI         SMA20         SMA50         EMA20  \
4608  1.514661  60.23

Calculation of log return of stock

R(t)=ln( P(t) / P(t-1) )

R->log return

P->closing price

mal by 100 at end to get percentage

In [8]:
import math
#can also use import numpy as np and np.log(value)
df["Log Return(%)"]=0.0
for i in range(1,len(df)):
  res=(df["Close"].iloc[i])/df["Close"].iloc[i-1]
  df.at[df.index[i], "Log Return(%)"]=(math.log(res))*100
print(df.iloc[4590:])

           Date     Adj Close         Close          High           Low  \
4590 2026-06-08  23123.000000  23123.000000  23267.300781  23070.150391   
4591 2026-06-09  23242.099609  23242.099609  23279.400391  23104.449219   
4592 2026-06-10  23214.949219  23214.949219  23425.349609  23184.599609   
4593 2026-06-11  23161.599609  23161.599609  23327.449219  23072.050781   
4594 2026-06-12  23622.900391  23622.900391  23645.349609  23313.900391   
4595 2026-06-15  23853.900391  23853.900391  24011.400391  23817.800781   
4596 2026-06-16  23989.150391  23989.150391  24002.599609  23888.199219   
4597 2026-06-17  24085.699219  24085.699219  24108.199219  23969.699219   
4598 2026-06-18  24168.000000  24168.000000  24189.250000  24036.949219   
4599 2026-06-19  24013.099609  24013.099609  24047.199219  23901.900391   
4600 2026-06-22  24102.900391  24102.900391  24168.050781  24073.150391   
4601 2026-06-23  23824.099609  23824.099609  24135.500000  23784.949219   
4602 2026-06-24  24021.65

Adding ATR

Create TR

TR=max(smth)

Then ATR14 for 14 days

then Wilder's Smoothing(like RSI)


In [9]:
import math
df["TR"]=0.0
for i in range(1,len(df)):
  df.at[df.index[i], "TR"]=max((df["High"].iloc[i]-df["Low"].iloc[i]), (abs(df["High"].iloc[i]-df["Close"].iloc[i-1])), abs(df["Close"].iloc[i-1]-df["Low"].iloc[i]))
ATRsum=0.0
for i in range(0,15):#15 coz ham tr of index 0 nhi compute kar rhe
  ATRsum=ATRsum+df["TR"].iloc[i]
ATR14=ATRsum/14
df["ATR"]=0.0
df.at[df.index[14], "ATR"]=ATR14
for i in range(15,len(df)):
  df.at[df.index[i], "ATR"]=((df['ATR'].iloc[i-1]*13)+df["TR"].iloc[i])/14
print(df.tail())


           Date     Adj Close         Close          High           Low  \
4608 2026-07-03  24270.849609  24270.849609  24378.150391  24252.349609   
4609 2026-07-06  24430.349609  24430.349609  24458.650391  24287.099609   
4610 2026-07-07  24398.699219  24398.699219  24530.900391  24348.949219   
4611 2026-07-08  23882.050781  23882.050781  24300.000000  23805.199219   
4612 2026-07-09  23962.800781  23962.800781  24134.699219  23925.699219   

              Open  Volume           U           D      avgUt  ...  \
4608  24375.650391  326600   95.150391    0.000000  84.658223  ...   
4609  24306.849609  329400  159.500000    0.000000  89.647675  ...   
4610  24464.449219  412600    0.000000   31.650391  83.671163  ...   
4611  24259.550781  420500    0.000000  516.648438  78.093086  ...   
4612  23928.949219       0   80.750000    0.000000  78.270213  ...   

             SMA20         SMA50         EMA20         EMA50        MACD  \
4608  23805.507324  23822.786914  23913.016100  2388

### Calculation of Bollinger Bands

Bollinger Bands are a technical indicator used to measure the volatility of the stock market.

They consist of three lines:

- Middle Band = 20-day Simple Moving Average (SMA20)
- Upper Band = SMA20 + (2 × Standard Deviation)
- Lower Band = SMA20 − (2 × Standard Deviation)

First, we calculate the 20-day Standard Deviation (STD20) of the closing prices.

The Standard Deviation tells us how much the stock price moves away from its average value.

Then, we calculate the Bollinger Upper Band using the formula:

Upper Band = SMA20 + (2 × STD20)

Similarly, we calculate the Bollinger Lower Band using the formula:

Lower Band = SMA20 − (2 × STD20)

When the distance between the Upper Band and Lower Band increases, it means the market is more volatile.

When the distance between the bands decreases, it means the market is less volatile.

Bollinger Bands help the model understand market volatility and price movement, which improves stock price prediction.

In [10]:
import math
df["STD20"] = 0.0
df["Bollinger Upper"] = 0.0
df["Bollinger Lower"] = 0.0
for i in range(19, len(df)):
    mean = df.at[df.index[i], "SMA20"]
    s = 0.0
    for j in range(i-19, i+1):
        diff = df.at[df.index[j], "Close"] - mean
        s += diff * diff
    variance = s / 20
    std = math.sqrt(variance)
    df.at[df.index[i], "STD20"] = std
    df.at[df.index[i], "Bollinger Upper"] = mean + (2 * std)
    df.at[df.index[i], "Bollinger Lower"] = mean - (2 * std)
print(df[["Close", "SMA20", "STD20","Bollinger Upper", "Bollinger Lower"]].tail())

             Close         SMA20       STD20  Bollinger Upper  Bollinger Lower
4608  24270.849609  23805.507324  365.855226     24537.217777     23073.796871
4609  24430.349609  23858.689844  375.387044     24609.463932     23107.915756
4610  24398.699219  23922.474805  352.654846     24627.784497     23217.165113
4611  23882.050781  23954.472363  316.666672     24587.805708     23321.139018
4612  23962.800781  23991.864941  267.466485     24526.797912     23456.931971
